# AI-WAF Logistic Regression training and evaluation

Run from a checkout of this repository. In Colab, mount Drive for persistent outputs and set the paths below to approved locations. Use an isolated Python environment with the pinned requirements. Preparation, training and evaluation start disabled.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO = Path.cwd().resolve()  # Set this to the checkout path in Colab.
assert (REPO / "requirements-training.txt").is_file(), "Set REPO to the checkout."
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# Run once in a fresh environment and restart the kernel if needed.
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt",
#                 "-r", "requirements-training.txt"], check=True)

STAGING_DIR = REPO / "data/staging"
SPLITS_DIR = REPO / "data/splits"
MODEL_DIR = REPO / "data/member3/models/lr"
DIAGNOSTIC_DIR = REPO / "data/member3/reports/calibration_validation"
FINAL_DIR = REPO / "data/member3/reports/independent_final"
PREPARE_CANDIDATE = False
TRAIN = False
RUN_DIAGNOSTICS = False
RUN_AUTHORIZED_FINAL = False

## Prepare the canonical candidate

Create the raw staging directory with prepare_raw_staging using the reviewed SR-BH, HTTPParams and optional baseline paths. Canonical preparation verifies staging checksums and writes train and validation candidates plus quarantine. Existing outputs are protected from overwrite. The held-out data stays locked. Use the team's separately approved canonical manifest and files to proceed.

In [ ]:
from training.week1_staging import prepare_canonical_if_ready, prepare_raw_staging

if PREPARE_CANDIDATE:
    # For first-time raw staging, run with approved source paths:
    # prepare_raw_staging(srbh_csv, httpparams_zip, STAGING_DIR, baseline_file=None)
    prepare_canonical_if_ready(STAGING_DIR, SPLITS_DIR)

## Verify approval, split integrity and dataset statistics

The shared loader checks approval, SHA-256, schema and disjoint request groups and canonical text. Review the class counts before fitting.

In [ ]:
from training.train_lr import load_splits

if TRAIN or RUN_DIAGNOSTICS or RUN_AUTHORIZED_FINAL:
    train_df, val_df = load_splits(SPLITS_DIR)
    for name, frame in (("train", train_df), ("calibration validation", val_df)):
        print(name, len(frame), frame["label"].value_counts().sort_index().to_dict())

## Fit TF-IDF and Logistic Regression, then calibrate and verify export

Fit TF-IDF on the training split and sigmoid calibration on the calibration-validation split. To reuse the existing trusted three-file export, leave TRAIN disabled and set MODEL_DIR to that export. Use a fresh path for a new export.

In [ ]:
from training.train_lr import calibrate, save_models, train_lr, train_tfidf
from training.evaluate import load_models

if TRAIN:
    vectorizer = train_tfidf(train_df["payload"])
    train_features = vectorizer.transform(train_df["payload"])
    model = train_lr(train_features, train_df["label"])
    val_features = vectorizer.transform(val_df["payload"])
    calibrator = calibrate(model, val_features, val_df["label"])
    save_models(vectorizer, model, calibrator, MODEL_DIR)

if TRAIN or RUN_DIAGNOSTICS or RUN_AUTHORIZED_FINAL:
    load_models(MODEL_DIR)  # Verify all three SHA-256 entries before loading trusted pickles.
    print("Verified model export:", MODEL_DIR)

## Calibration-validation and authorized final evaluation

Calibration-validation diagnostics use the partition that fitted the calibrator. Enable RUN_AUTHORIZED_FINAL only after team authorization. Final evaluation also requires an approved canonical manifest with final evaluation authorization and a disjoint checksummed test split. Both modes use the frozen configured threshold and write the same metrics and figures to separate report directories.

In [ ]:
import json

evaluation_command = [
    sys.executable, "-m", "training.evaluate",
    "--splits-dir", str(SPLITS_DIR), "--models-dir", str(MODEL_DIR),
]
if RUN_DIAGNOSTICS:
    subprocess.run(
        evaluation_command + ["--validation-diagnostics", "--output-dir", str(DIAGNOSTIC_DIR)],
        check=True,
    )
    print("Calibration-validation reports:", DIAGNOSTIC_DIR)

if RUN_AUTHORIZED_FINAL:
    manifest = json.loads((SPLITS_DIR / "canonical_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("final_evaluation_authorized") is not True:
        raise RuntimeError("Final evaluation authorization is required.")
    subprocess.run(evaluation_command + ["--output-dir", str(FINAL_DIR)], check=True)
    print("Independent final reports:", FINAL_DIR)